In [ ]:
import pandas as pd
import re
import os

# -------------------------------------------------------------
# 1. Import the Excel File
# -------------------------------------------------------------
excel_path = r"F:\Chimney Work\Parivesh Work\Leads 2025-26.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Read the Excel file into a DataFrame
df = pd.read_excel(excel_path)

if 'Project Details XML' not in df.columns:
    raise KeyError("The Excel file does not contain a column named 'Project Details XML'")

print(f"Successfully loaded {len(df)} rows. Extracting contacts...")

# -------------------------------------------------------------
# 2. Compile Regular Expressions for Contact Extraction
# -------------------------------------------------------------
# Matches standard email formatting structures inside raw markup text
email_regex = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

# Matches continuous strings of digits to capture phone numbers
digit_regex = re.compile(r'\b\d{6,12}\b')

# Initialize the new placeholder target columns with "N/A"
for i in range(1, 3):
    df[f'Email_{i}'] = "N/A"

for i in range(1, 4):
    df[f'Mobile_{i}'] = "N/A"
    df[f'Landline_{i}'] = "N/A"

# -------------------------------------------------------------
# 3. Extraction Loop
# -------------------------------------------------------------
for idx, row in df.iterrows():
    xml_data = str(row['Project Details XML'])
    
    # Skip processing if the row is missing or contains fallback string tags
    if pd.isna(row['Project Details XML']) or xml_data == "N/A" or xml_data.strip() == "":
        continue
        
    # --- A. Extract Unique Emails ---
    # Find matches and deduplicate while preserving original order
    found_emails = list(dict.fromkeys(email_regex.findall(xml_data)))
    # Strip out common design asset file-extensions picked up by broad matches
    found_emails = [e for e in found_emails if not e.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.pdf'))]
    
    for i in range(min(2, len(found_emails))):
        df.at[idx, f'Email_{i+1}'] = found_emails[i]
        
    # --- B. Extract & Classify Numbers ---
    raw_numbers = list(dict.fromkeys(digit_regex.findall(xml_data)))
    
    mobiles = []
    landlines = []
    
    for num in raw_numbers:
        # Indian Mobile Format Validation: Exactly 10 digits beginning with 6, 7, 8, or 9
        if len(num) == 10 and num[0] in '6789':
            mobiles.append(num)
        # Landline Format Validation: Lengths 6 to 11 digits (with or without STD prefixes)
        elif 6 <= len(num) <= 11 and num not in mobiles:
            # Drop false-positive matches belonging to common layout years
            if num not in ['2024', '2025', '2026']:
                landlines.append(num)

    # Distribute collected numbers up to the specified limits
    for i in range(min(3, len(mobiles))):
        df.at[idx, f'Mobile_{i+1}'] = mobiles[i]
        
    for i in range(min(3, len(landlines))):
        df.at[idx, f'Landline_{i+1}'] = landlines[i]

# -------------------------------------------------------------
# 4. Save the New Columns Back to the Original File
# -------------------------------------------------------------
print("Saving all extracted details back into the original Excel sheet...")
df.to_excel(excel_path, index=False)
print(f"🎉 Task complete! File successfully updated at: {excel_path}")